In [2]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.contrib.piecewise import PiecewiseLinearFunction
from pyomo.opt import SolverFactory
from pyomo.core.base import TransformationFactory
from pyomo.opt import SolverStatus, TerminationCondition
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Tuple
import numpy as np
import math
import matplotlib.pyplot as plt
import bisect
import itertools as it
from tqdm import tqdm   # 先 import
#from pyomo.environ import ConcreteModel, Var, Constraint, Expression

def evaluate_Q_at(model, first_stg_vars, first_stg_vals, solver):
    """
    Given y = y_val , minimize obj_expr and return v(y).
    This function temporarily increments the objective and clears it after completion, without changing the model structure.
    """
    # Clear any remaining As/pw/obj (to prevent it from being left over from the previous round)
    del_components(model)
    
    for u, v in zip(first_stg_vars, first_stg_vals):
        u.fix(value(v))
    model.obj = Objective(expr=model.obj_expr, sense=minimize)
    results = solver.solve(model, tee=False)

    status_ok = (results.solver.status == SolverStatus.ok)
    term_ok = (results.solver.termination_condition == TerminationCondition.optimal)
    if not (status_ok and term_ok):
        # check if solution okay
        raise RuntimeError(f"Scenario evaluate at y={first_stg_vals} not optimal: "
                           f"status={results.solver.status}, term={results.solver.termination_condition}")

    v_opt = value(model.obj_expr)
    # clear temporarily objective
    model.del_component('obj')
    for u in first_stg_vars:
        u.unfix()
    return v_opt

def _fmt_item(x, prec=6):
    if isinstance(x, (tuple, list)):
        return "(" + ", ".join(f"{float(v):.{prec}f}" for v in x) + ")"
    try:
        return f"{float(x):.{prec}f}"
    except Exception:
        return str(x)
    
import pyomo.environ as pyo

################################################################################################
############################# warm start #######################################################
################################################################################################

def dump_solution(m):
    """从模型 m 提取解，用于下次温启动。按你的变量名收集。"""
    return {
        'Kp': pyo.value(m.Kp),
        'Ki': pyo.value(m.Ki),
        'Kd': pyo.value(m.Kd),
        'x': {t: pyo.value(m.x[t]) for t in m.T},
        'u': {t: pyo.value(m.u[t]) for t in m.T},
        'e': {t: pyo.value(m.e[t]) for t in m.T},
        'I': {t: pyo.value(m.I[t]) for t in m.T},
    }

def apply_warm_values(m, sol):
    """把 sol 写回到 m.Var.value（若 sol 为 None 就跳过）。"""
    if not sol:
        return
    if sol.get('Kp') is not None: m.Kp.value = sol['Kp']
    if sol.get('Ki') is not None: m.Ki.value = sol['Ki']
    if sol.get('Kd') is not None: m.Kd.value = sol['Kd']
    for t in m.T:
        if 'x' in sol and t in sol['x']: m.x[t].value = sol['x'][t]
        if 'u' in sol and t in sol['u']: m.u[t].value = sol['u'][t]
        if 'e' in sol and t in sol['e']: m.e[t].value = sol['e'][t]
        if 'I' in sol and t in sol['I']: m.I[t].value = sol['I'][t]

def push_start_to_gurobi(opt, m):
    """
    把 m 中的 Var.value 同步为 Gurobi 的 Start（真正让求解器用作温启动）。
    需要 gurobi_persistent 求解器。
    """
    for v in m.component_data_objects(pyo.Var, active=True):
        val = v.value
        if val is None:
            continue  # 第一次没有起点就跳过，绝不会报错
        # 对整数/二进制变量做轻微修正，避免起点被拒
        if v.is_binary() or v.is_integer():
            # 挨得很近就四舍五入；否则让 Gurobi 自修复也行
            if abs(val - round(val)) < 1e-6:
                val = int(round(val))
        # 裁剪到边界内
        if v.has_lb() and val < v.lb: val = v.lb
        if v.has_ub() and val > v.ub: val = v.ub
        opt.set_var_attr(v, "Start", float(val))

################################################################################################
################################################################################################
################################################################################################

def print_nodes_row(existing_nodes, new_node,
                    existing_values=None, new_value=None,
                    prec_node=2, prec_value=6, pad=2, label_new="(new node)",
                    highlight_min=True):
    """打印节点与值表，最小值高亮红色"""

    headers = [f"node{i+1}" for i in range(len(existing_nodes))] + [f"node{len(existing_nodes)+1} {label_new}"]
    node_strs = [_fmt_item(n, prec_node) for n in existing_nodes] + [_fmt_item(new_node, prec_node)]

    have_vals = existing_values is not None or new_value is not None
    if existing_values is None:
        existing_values = [None] * len(existing_nodes)

    val_strs = []
    if have_vals:
        for v in existing_values:
            val_strs.append(_fmt_item(v, prec_value) if v is not None else "")
        val_strs.append(_fmt_item(new_value, prec_value) if new_value is not None else "")

    # 找到最小值
    min_val = None
    if have_vals and highlight_min:
        try:
            nums = [float(v) for v in existing_values if v is not None]
            if new_value is not None:
                nums.append(float(new_value))
            if nums:
                min_val = min(nums)
        except Exception:
            pass

    # 计算列宽
    cols = max(len(headers), len(node_strs))
    widths = []
    for j in range(cols):
        pieces = []
        if j < len(headers):   pieces.append(headers[j])
        if j < len(node_strs): pieces.append(node_strs[j])
        if have_vals and j < len(val_strs): pieces.append(val_strs[j])
        w = max(len(s) for s in pieces) + pad*2
        widths.append(w)

    def _center(s, w): return s.center(w)
    def _right(s, w):  return s.rjust(w)

    header_line = "".join(_center(h, widths[i]) for i, h in enumerate(headers))
    sep_line = "".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep_line)

    node_line = "".join(_right(s, widths[i]) for i, s in enumerate(node_strs))
    print(node_line)

    if have_vals:
        # 给最小值上色
        val_line_parts = []
        for i, s in enumerate(val_strs):
            if s and min_val is not None and abs(float(s) - min_val) < 1e-12:
                colored = f"\033[31m{s}\033[0m"  # 红色
                val_line_parts.append(_right(colored, widths[i] + 9))  # 留点额外空间
            else:
                val_line_parts.append(_right(s, widths[i]))
        print("".join(val_line_parts))

def del_components(model):
    for comp in ['obj', 'As', 'pw', 'pw_fun', 'pw_As', 'pw_link']:
        if hasattr(model, comp):
            model.del_component(comp)

def corners_from_bounds(firt_stg_vars):
    """给一组 Pyomo Var 生成所有 box 角点（每维取 lb/ub）"""
    bounds = []
    for y in firt_stg_vars:
        lb, ub = y.lb, y.ub
        if lb is None or ub is None:
            raise ValueError(f"{y.name} 缺少上下界，无法生成角点")
        bounds.append((float(lb), float(ub)))
    # 每维挑 lb/ub 的笛卡尔积
    return list(it.product(*[(lb, ub) for (lb, ub) in bounds]))


def add_nd_piecewise(
    model,
    firt_stg_vars,                 # [x1, x2, ..., xN]  模型里的 first stage Var
    points,                 # [(c11,...,c1N), (c21,...,c2N), ...]  所有节点（同维度 N）
    values,                 # 与 points 对齐的一维 list/array，或 dict{point_tuple: value}
    name="pw",
    relation="==",          # '==', '>='(下界/内逼近), '<='(上界/外逼近)
    round_ndigits=12,       # 为避免浮点比较问题，对坐标做轻微 round
):
    """
    返回 (z, pw)。z 是 Var（或你可将 make_z_var=False 改成返回 Expression）。
    """

    # 维度检查
    if len(points) == 0:
        raise ValueError("points 不能为空")
    N = len(firt_stg_vars)
    for pt in points:
        if len(pt) != N:
            raise ValueError(f"points 中出现与 x_vars 维度不一致的点: {pt}")
        
    del_components(model)

    # 统一坐标的浮点表示，避免查表时精度问题
    def keyize(coords):
        return tuple(round(float(c), round_ndigits) for c in coords)

    norm_points = [keyize(pt) for pt in points]

    # 把 values 统一成 dict 表
    if isinstance(values, dict):
        table = {keyize(k): float(v) for k, v in values.items()}
        # 确保每个 point 都有值
        miss = [pt for pt in norm_points if pt not in table]
        if miss:
            raise KeyError(f"values 缺少这些点的取值: {miss[:5]}{' ...' if len(miss)>5 else ''}")
    else:
        # 视作与 points 对齐的一维序列
        if len(values) != len(points):
            raise ValueError("values 长度应与 points 数量一致（或传 dict）")
        table = {pt: float(v) for pt, v in zip(norm_points, values)}

    # 查表函数（仅在节点上被调用）
    def _f_from_table(*coords):
        return table[keyize(coords)]

    # 创建并挂到模型
    pw = PiecewiseLinearFunction(points=norm_points, function=_f_from_table, name=f"{name}_fun")
    model.add_component(pw.name, pw)

    # 组成表达式
    pw_expr = pw(*firt_stg_vars)

    As = Var(name=f"{name}_As")
    model.add_component(As.name, As)
    if relation == "==":
        link = Constraint(expr=As == pw_expr)
    elif relation == ">=":
        link = Constraint(expr=As >= pw_expr)
    elif relation == "<=":
        link = Constraint(expr=As <= pw_expr)
    else:
        raise ValueError("relation 只能是 '==', '>=', '<='")
    model.add_component(f"{name}_link", link)

    TransformationFactory('contrib.piecewise.convex_combination').apply_to(model)
        
    return As, pw

def clone_and_get_vars(m_old, first_stage_vars):
    """
    克隆模型，并按名字列表返回新模型中的变量组件
    var_names: ['y', 'x', ...]  (只能是容器名，不是元素)
    """
    m_new = m_old.clone()
    first_stage_vars_new = []
    for v in first_stage_vars:
        v_new = m_new.find_component(v.name)
        if v_new is None:
            raise KeyError(f"在新模型里找不到变量 '{v.name}'")
        first_stage_vars_new.append(v_new)
    return m_new, first_stage_vars_new

# delete repeated nodes
def unique_points(points, atol=1e-9):
    out = []
    for p in points:
        if not any(all(abs(a-b) <= atol for a,b in zip(p, q)) for q in out):
            out.append(p)
    return out

# underestimator code
def nc_underest(model_list, first_stg_vars_list, m_tmpl_list, target_nodes, picture_shown=False, v_list=False, tolerance=1e-8, probs=None):
    """
    Parameters:
        #bounds (list): contains 2 float which is lower and upper bound of variable
        model_list (list): model with submodels corresponds to each scenario
        first_stg_var (list): 
        m_tmpl_list (list): [template model, template model first stg variables list]
        target_nodes (float): number of target nodes
        tolerance (float): decide when to stop

    Returns: delta (float): delta
             errors (float): hausdorff error
             y_nodes (list): y node (to make plot)
             as_nodes_list[0] (list): As node value (to make plot)
             ms_list[0] (float): ms for first scenario (to make plot)
    """
    N = len(model_list)
    if probs is None:
        probs = [1.0]*N
    assert len(probs) == N
    UB_best = float('inf')
    y_best  = None
    as_nodes_list = [[] for _ in range(N)]
    ms_list = [None] * N
    new_nodes_list = [None] * N # Storing potential new nodes
    As_min_list = []
    under_tol = 1e-8
    add_node_history = []
    # set up solver
    solver = SolverFactory('gurobi')
    solver.options.update({
    'MIPGap': 1e-2,       # 先 1e-3 或 1e-2，稳定后再收紧
    # 'MIPGapAbs': 不设
    'FeasibilityTol': 1e-6,
    'IntFeasTol':     1e-6,
    'OptimalityTol':  1e-6,
    'NumericFocus':   1,   # 除非数值告警，再升到 2
    'Presolve':       2,
    # 'Method'/'Crossover' 不写，默认自动
    'NonConvex':      2,   # 确有双线性才保留
    'MIPFocus':       1,   # 先找可行优质解
    # 'TimeLimit':     60  # 可做阶段性比较
})

    '''    
    solver.options.update({
        'MIPGap': 1e-4,              
        'FeasibilityTol': 1e-6,  
        'IntFeasTol':     1e-9,  
        'OptimalityTol': 1e-6,
        'NumericFocus': 2,      
        'ScaleFlag':    1,       
        'Presolve': 2,              
        'NonConvex': 2, 
    })
    '''

    # start from corner nodes
    first_stg_nodes = corners_from_bounds(first_stg_vars_list[0])
    for i in range(N):
        as_nodes_list[i].extend(
            evaluate_Q_at(model_list[i], first_stg_vars_list[i], node, solver) for node in first_stg_nodes
        )
    print('corner nodes are ', first_stg_nodes)
    print('as_nodes_list are ', as_nodes_list)

    if target_nodes <= len(first_stg_nodes):
        print('target_nodes number should be larger than ',len(first_stg_nodes))
        return

    print('Start from ',len(first_stg_nodes),' corner nodes')
    print('The goal is to get ',target_nodes,' nodes')
    k_list = []

    for k in tqdm(range(len(first_stg_nodes)+1, target_nodes+1), desc="Adding nodes"):
        print('##################################################')
        print('##################################################')
        print('Start adding node ',k)
        k_list.append(k)
        for i in range(N):
            print(' ')
            print('Solving scenario ',i)
            # define piecewise function for each scenario
            del_components(model_list[i])

            # warm start

            # ---- 修复后 ----
            '''
            for j, m_warm in enumerate(model_list):
                apply_warm_values(m_warm, warm_solutions[j])   # 用 j
                opt = persistent_solvers[j]
                opt.set_instance(m_warm)                        # 重新绑定实例
                push_start_to_gurobi(opt, m_warm)
                _ = opt.solve(tee=False)                        # 这里仅用来刷新可行起点（可选）
                warm_solutions[j] = dump_solution(m_warm)
            '''
            for j, m_warm in enumerate(model_list):
                apply_warm_values(m_warm, warm_solutions[j])
                # opt = persistent_solvers[j]
                # opt.set_instance(m_warm)
                # push_start_to_gurobi(opt, m_warm)
                # _ = opt.solve(tee=False)  # ← 删除
                warm_solutions[j] = dump_solution(m_warm)





            # start solving sceanario
            As, _ = add_nd_piecewise(
            model_list[i], first_stg_vars_list[i], first_stg_nodes, as_nodes_list[i],
            name="pw", relation="=="
            )
            model_list[i].obj = Objective(expr=model_list[i].obj_expr - As, sense=minimize)
            #results = SolverFactory("gurobi").solve(model_list[i], tee=False)
            results = solver.solve(model_list[i], tee=True)
            if (results.solver.status != SolverStatus.ok) or \
               (results.solver.termination_condition != TerminationCondition.optimal):
                print("⚠ There may be problems with the solution")
                
            ms_list[i] = value(model_list[i].obj)
            # insert new nodes
            new_nodes_list[i] = tuple(value(v) for v in first_stg_vars_list[i])
            print('new node is ',tuple(value(v) for v in first_stg_vars_list[i]))
            print('ms is ',value(model_list[i].obj))

        # define and solve the sum model to check if error at possible new node is too big
        arr = np.array(as_nodes_list, dtype=float, ndmin=2)  
        assum_nodes = arr.sum(axis=0) 

        print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
        print(first_stg_nodes)
        print(assum_nodes)

        # build As_sum model and solve for possible node pf max error
        model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
        del_components(model_sum)
        As, pw = add_nd_piecewise(
        model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
        name="pw", relation="=="
        )
        model_sum.obj = Objective(expr= As, sense=minimize)
        results = solver.solve(model_sum, tee=True)
        if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
            pass
        else:
            print("Sum model doesn't get solved normally")

        # get the output
        As_min = results.problem.lower_bound
        print(f'As_min at possible new node is {As_min}')
        node_star = tuple(value(v) for v in model_sum_first_stg_vars) 

        if (node_star is None) or (node_star in first_stg_nodes):
            avg = []
            for j in range(len(first_stg_nodes[0])):
                comp_vals = [node[j] for node in first_stg_nodes]
                avg.append(sum(comp_vals) / len(comp_vals))
            node_star = tuple(avg)
            As_min = value(pw(*node_star))
        q_node_star = 0
        for i in range(N):
            q_node_star += evaluate_Q_at(model_list[i], first_stg_vars_list[i], node_star, solver)
        print(f'Real value at possible new node is {q_node_star}')
        errors_node_star = abs(As_min - q_node_star)
        print(f'error at possible new node is {errors_node_star}')

        sum_ms = sum(ms_i for ms_i in ms_list)
        
        print('Sum *****************************************')
        print('error at y_star is ',errors_node_star)
        print('y_star is ',node_star)
        print('ms_list and sum_ms is ',ms_list,sum_ms)
        if errors_node_star > abs(sum_ms):
            new_node = node_star
            print('new node choosen from error')
        else:
            min_index = np.argmin(ms_list)
            new_node = new_nodes_list[min_index]
            print('new node choosen from ms')
        As_min_list.append(As_min+sum_ms)
        add_node_history.append(new_node)
        print_nodes_row(first_stg_nodes, new_node,
                existing_values=assum_nodes,
                new_value=q_node_star,
                prec_node=1,
                prec_value=6)
        print('new node is',new_node)
        print('Current As_min is',As_min_list[-1])
        print('*****************************************')
        print('')
        #######################################################              

        print('current nodes are ', first_stg_nodes)
        print('as_nodes_list are ', as_nodes_list)
        print('new_node is', new_node)
        first_stg_nodes.append(new_node)
        for i in range(N):
            as_nodes_list[i].append(evaluate_Q_at(model_list[i], first_stg_vars_list[i], new_node, solver))
        arr = np.array(as_nodes_list, dtype=float, ndmin=2)  
        assum_nodes = arr.sum(axis=0) 

        idx = int(np.argmin(assum_nodes))        
        val = assum_nodes[idx]                   
        print(f"\033[31mcurrent node is {idx+1} node, value is {val:.6f}\033[0m")
        print(f"node is{first_stg_nodes[idx]}")

    # define and solve the sum model
    arr = np.array(as_nodes_list, dtype=float, ndmin=2)  
    assum_nodes = arr.sum(axis=0) 

    # build As_sum model and solve for possible node pf max error
    model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
    del_components(model_sum)
    As, _ = add_nd_piecewise(
    model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
    name="pw", relation="=="
    )
    model_sum.obj = Objective(expr= As, sense=minimize)
    results = solver.solve(model_sum, tee=True)
    if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
        pass
    else:
        print("Sum model doesn't get solved normally")

    # get the output
    output_lb = results.problem.lower_bound + sum(ms_list)
    print('lower bound is ',output_lb)
    print('node is ',tuple(value(v) for v in model_sum_first_stg_vars) )

    return output_lb, first_stg_nodes, [k_list, As_min_list, add_node_history]

##################################################################################################################
##################################################################################################################
##################################################################################################################

def build_pid_model(
    T=10,                 # 时间步数
    h=0.2,                # 步长 Δt
    scen=None,            # 单个场景数据: {"Ku":..., "tau":..., "d":[...], "sp":[...]}
    weights=(1.0, 0.01),  # 目标权重 (w_e, w_u)
    bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,100), "Kd":(0,100)},
    use_cvar=False,       # 保留参数，但单场景下一般不需要
    alpha=0.95
):
    """
    单场景 PID 控制模型:
      tau * x_dot + x = Ku * u + d
      u = Kp*e + Ki*I + Kd*de/dt,   e = sp - x,   I_dot = e
    离散化: 隐式欧拉
    """
    assert scen is not None, "请提供一个场景字典"

    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]

    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    # 一阶段变量 (PID 参数)
    m.Kp = pyo.Var(bounds=bounds["Kp"])
    m.Ki = pyo.Var(bounds=bounds["Ki"])
    m.Kd = pyo.Var(bounds=bounds["Kd"])

    # 二阶段变量
    m.x = pyo.Var(m.T, bounds=bounds["x"])
    m.u = pyo.Var(m.T, bounds=bounds["u"])
    m.e = pyo.Var(m.T, bounds=bounds["e"])
    m.I = pyo.Var(m.T, bounds=bounds["I"])


    # 误差 e_t = sp_t - x_t
    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    # I 动态
    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    # 系统动态 (隐式欧拉)
    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # PID 控制律
    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    # 初值
    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    # 成本函数 (单场景就是一个值)
    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(
        h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T
    ))

    # 目标: 单场景情况下就直接最小化 cost
    #m.obj = pyo.Objective(expr=m.cost, sense=pyo.minimize)
    m.obj_expr = pyo.Expression(expr = m.cost)

    return m,  [m.Kp, m.Ki, m.Kd]


In [3]:
if __name__ == '__main__':
    T, h = 10, 0.2
    times = [t for t in range(T+1)]

    def step_sp(t): return 1.0 if t*h >= 0.5 else 0.0
    #def make_d(amp): return [amp*math.sin(0.5*t*h) for t in times]
    def make_d(amp, bias=0.1): 
        return [bias + amp*math.sin(0.5*t*h) for t in times]

    '''scenarios = {
        1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
    }'''
    scenarios = {
        1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
        2: {"prob": 0.4, "Ku": 2.7, "tau": 1.8, "d": make_d(0.5), "sp": [step_sp(t) for t in times]},
        3: {"prob": 0.2, "Ku": 3.3, "tau": 2.2, "d": make_d(0.8), "sp": [step_sp(t) for t in times]},
    }
    probs = [sc.get("prob", 1.0) for _, sc in scenarios.items()]
    model_list = []
    first_stg_vars_list = []
    I_max = 1.5 * T*h*21  
    #bounds={"x":(-20,20), "u":(-20,20), "e":(-21,21), "I":(-I_max,I_max), "Kp":(0,100), "Ki":(0,100), "Kd":(0,100)}
    bounds={"x":(-20,20), "u":(-20,20), "e":(-21,21), "I":(-I_max,I_max), "Kp":(0,50), "Ki":(0,100), "Kd":(0,50)}
    for _, scen in scenarios.items():
        m, m_f_stg_vars = build_pid_model(T=T, h=h, scen=scen, weights=(1.0, 0.09), bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(m_f_stg_vars)
    m_tmpl = pyo.ConcreteModel()
    m_tmpl.T = pyo.RangeSet(0, T)
    m_tmpl.Tm = pyo.RangeSet(1, T)
    m_tmpl.Kp = pyo.Var(bounds=bounds["Kp"])
    m_tmpl.Ki = pyo.Var(bounds=bounds["Ki"])
    m_tmpl.Kd = pyo.Var(bounds=bounds["Kd"])
    m_tmpl_list = [m_tmpl, [m_tmpl.Kp, m_tmpl.Ki, m_tmpl.Kd]]


    # 为每个模型各自创建一个 persistent 实例（也可复用一个实例按需 set_instance）
    opts = {
        'MIPGap': 1e-2,       # 先用 1% 提速外层；收尾再收紧
        'FeasibilityTol': 1e-3,
        'IntFeasTol':     1e-3,
        'OptimalityTol':  1e-3,
        'NumericFocus':   1,
        'Presolve':       2,
        'NonConvex':      2,  # 你有双线性
        'MIPFocus':       1,
        # 可选：'TimeLimit': 10
    }

    persistent_solvers = []
    for m in model_list:
        opt = SolverFactory("gurobi_persistent")
        opt.set_instance(m)
        for k, v in opts.items():
            opt.set_gurobi_param(k, v)
        persistent_solvers.append(opt)

    # 给每个场景准备一份温启动“解的存档”
    warm_solutions = [None for _ in model_list]

    nc_underest(model_list, first_stg_vars_list, m_tmpl_list, target_nodes=15,
                    picture_shown=False, v_list=False, tolerance=1e-8, probs=probs
    )

Set parameter MIPGap to value 0.01
Set parameter FeasibilityTol to value 0.001
Set parameter IntFeasTol to value 0.001
Set parameter OptimalityTol to value 0.001
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Set parameter MIPGap to value 0.01
Set parameter FeasibilityTol to value 0.001
Set parameter IntFeasTol to value 0.001
Set parameter OptimalityTol to value 0.001
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Set parameter MIPGap to value 0.01
Set parameter FeasibilityTol to value 0.001
Set parameter IntFeasTol to value 0.001
Set parameter OptimalityTol to value 0.001
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
corner nodes are  [(0.0, 0.0, 0.0), (0.0, 0.0, 50.0), (0.0, 100.0, 0.0), (0.0, 100.0, 50.0)

Adding nodes:   0%|          | 0/7 [00:00<?, ?it/s]

##################################################
##################################################
Start adding node  9
 
Solving scenario  0
Read LP format model from file C:\Users\Administrator\AppData\Local\Temp\tmp23t1hz9x.pyomo.lp
Reading time = 0.00 seconds
x1: 46 rows, 63 columns, 142 nonzeros
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (22631.2))

CPU model: AMD Ryzen 9 7900X 12-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Optimize a model with 46 rows, 63 columns and 142 nonzeros
Model fingerprint: 0xc93eea81
Model has 22 quadratic objective terms
Model has 1

Adding nodes:  14%|█▍        | 1/7 [00:23<02:22, 23.81s/it]

current node is 9 node, value is 0.547364
node is(5.565277980439508e-07, 2.98713865633997, 1.4935684299938679)
##################################################
##################################################
Start adding node  10
 
Solving scenario  0
Read LP format model from file C:\Users\Administrator\AppData\Local\Temp\tmpb1al2h9u.pyomo.lp
Reading time = 0.00 seconds
x1: 60 rows, 85 columns, 235 nonzeros
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (22631.2))

CPU model: AMD Ryzen 9 7900X 12-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Optimize a model with 60 ro

...

Adding nodes:  86%|████████▌ | 6/7 [00:58<00:07,  7.45s/it]

current node is 11 node, value is 0.478655
node is(2.948545704542877, 0.0, 0.07936592149218069)
##################################################
##################################################
Start adding node  15
 
Solving scenario  0
Read LP format model from file C:\Users\Administrator\AppData\Local\Temp\tmp8hpa6xrp.pyomo.lp
Reading time = 0.01 seconds
x1: 150 rows, 257 columns, 1071 nonzeros
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (22631.2))

CPU model: AMD Ryzen 9 7900X 12-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Optimize a model with 150 rows, 257 col

Adding nodes: 100%|██████████| 7/7 [01:05<00:00,  9.38s/it]

current node is 11 node, value is 0.478655
node is(2.948545704542877, 0.0, 0.07936592149218069)


Read LP format model from file C:\Users\Administrator\AppData\Local\Temp\tmpqsydqw1k.pyomo.lp
Reading time = 0.00 seconds
x1: 22 rows, 39 columns, 170 nonzeros
Set parameter MIPGap to value 0.01
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (22631.2))

CPU model: AMD Ryzen 9 7900X 12-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Optimize a model with 22 rows, 39 columns and 170 nonzeros
Model fingerprint: 0x1a2d4957
Variable types: 20 continuous, 19 integer (19 binary)
Coefficient statistics:
  Matrix range     [6e-07, 1e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 8e+06]
  RHS range    

In [19]:
solver = pyo.SolverFactory("ipopt")
first_stg_nodes = [(0.0, 0.0, 0.0), (0.0, 0.0, 1000.0), (0.0, 1000.0, 0.0), (0.0, 1000.0, 1000.0), (100.0, 0.0, 0.0), (100.0, 0.0, 1000.0), (100.0, 1000.0, 0.0), (100.0, 1000.0, 1000.0)]
assum_nodes = [4.54023507, 0.05320473, 0.04886529, 0.05321642, 0.04795115, 0.05320593, 0.05042764, 0.05321691]

# build As_sum model and solve for possible node pf max error
model_sum, model_sum_first_stg_vars = clone_and_get_vars(m_tmpl_list[0], m_tmpl_list[1])
del_components(model_sum)
As, pw = add_nd_piecewise(
model_sum, model_sum_first_stg_vars, first_stg_nodes, assum_nodes,
name="pw", relation="=="
)
model_sum.obj = Objective(expr= As, sense=minimize)

#results = SolverFactory("gurobi").solve(model_sum, tee=False)
results = solver.solve(model_sum, tee=False)
if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    pass
else:
    print("Sum model doesn't get solved normally")

# get the output
As_min = results.problem.lower_bound
print(f'As_min at possible new node is {As_min}')
node_star = tuple(value(v) for v in model_sum_first_stg_vars) 

As_min at possible new node is -inf


In [20]:
print(f'As_min at possible new node is {value(model_sum.obj)}')

As_min at possible new node is 0.04795113247498382
